# 02 — Extract BOM

| | |
|---|---|
| **Pipeline stage** | `extract_bom` |
| **Service module** | `backend.services.scraper.scrape_project` (via `pipeline.scrape_makerworld`) |
| **Website equivalent** | `extract_bom` events in `POST /api/import/stream` |
| **Prev / Next** | `01_scrape.ipynb` → `03_parse_bom.ipynb` |

Saves BOM bytes to `data/` or caches embedded parts for offline notebook runs. Uses `safe_scrape()` (same crawl, 90s timeout).


In [ ]:
from backend.notebook_utils import (
    DATA_DIR,
    cache_parts,
    cache_scrape_summary,
    notebook_progress,
    pick_sample_url,
    prepare_crawl_env,
    safe_scrape,
)

prepare_crawl_env(scraper="auto")
DATA_DIR.mkdir(exist_ok=True)
progress = notebook_progress("02")

SAMPLE_URL = pick_sample_url(index=1, require_bom=True)["url"]
result = await safe_scrape(SAMPLE_URL, on_progress=progress, timeout_s=90)

if result is None:
    print("Notebooks 03+ can use data/sample_bom.csv offline")
elif result.bom_bytes and result.bom_filename:
    out = DATA_DIR / result.bom_filename
    out.write_bytes(result.bom_bytes)
    print(f"Saved file BOM → {out}")
    cache_scrape_summary(result)
elif result.embedded_parts:
    cache_parts(result.embedded_parts)
    cache_scrape_summary(result)
    print(f"Cached {len(result.embedded_parts)} embedded parts → data/notebook_parts.json")
    for part in result.embedded_parts[:8]:
        print(f"  {part.quantity:g}x {part.original_name}")
else:
    print("No BOM on this model")


## Description BOM — quantity & hardware keywords

Projects without an embedded BOM file may list hardware in the **project description** (`bom_source == "description"`). The cell below uses the same rule-based parser as the API (`parsers/makerworld/description.py`) and tags each line with signals from `parsers/helpers/hardware_signals.py` — metric sizes (M3–M12), fastener type/prefix/suffix, `30mm` lengths, and axial dims like `5x30`.

In [ ]:
from backend.notebook_utils import REPO_ROOT
from backend.services.parsers.makerworld.description import (
    extract_candidate_lines,
    parse_bom_line,
    parts_from_description,
)
from backend.services.parsers.helpers.hardware_signals import (
    extract_metric_sizes,
    has_axial_dimension,
    has_fastener_prefix,
    has_fastener_suffix,
    has_fastener_type,
    has_hardware_signal,
    has_length_mm,
    has_metric_fastener,
)

MEGA_PYTHON_FIXTURE = REPO_ROOT / "tests/fixtures/description_mega_python.txt"

if result is not None and result.description.strip():
    description_text = result.description
    source_label = result.makerworld_url or SAMPLE_URL
elif MEGA_PYTHON_FIXTURE.is_file():
    description_text = MEGA_PYTHON_FIXTURE.read_text(encoding="utf-8")
    source_label = str(MEGA_PYTHON_FIXTURE.relative_to(REPO_ROOT))
else:
    description_text = ""
    source_label = ""

if not description_text.strip():
    print("No description text — run the scrape cell above or add tests/fixtures/description_mega_python.txt")
else:
    print(f"Description source: {source_label}\n")

    lines, from_section = extract_candidate_lines(description_text)
    parts = parts_from_description(description_text)
    print(
        f"Candidate lines: {len(lines)}  |  Parsed parts: {len(parts)}  "
        f"|  Explicit BOM section: {from_section}"
    )
    if result is not None and result.description_parts:
        print(f"Scrape description_parts: {len(result.description_parts)}")
    print()

    def hardware_tags(text: str) -> list[str]:
        tags: list[str] = []
        if has_metric_fastener(text):
            tags.append("metric_fastener")
        sizes = extract_metric_sizes(text)
        if sizes:
            tags.append("metric:" + ",".join(sizes))
        if has_fastener_type(text):
            tags.append("fastener_type")
        if has_fastener_prefix(text):
            tags.append("fastener_prefix")
        if has_fastener_suffix(text):
            tags.append("fastener_suffix")
        if has_length_mm(text):
            tags.append("length_mm")
        if has_axial_dimension(text):
            tags.append("axial_dim")
        if has_hardware_signal(text) and not tags:
            tags.append("hardware_other")
        return tags or ["—"]

    print(f"{'QTY':>6}  {'KEYWORDS':<40}  LINE")
    print("-" * 110)
    for line in lines[:50]:
        part = parse_bom_line(line, require_hardware_keyword=not from_section)
        inspect_text = line
        qty_display = "—"
        if part:
            inspect_text = f"{part.original_name} {part.specification}".strip()
            qty_display = f"{part.quantity:g}"
        short_line = line if len(line) <= 58 else line[:57] + "…"
        print(f"{qty_display:>6}  {', '.join(hardware_tags(inspect_text)):<40}  {short_line}")

    if len(lines) > 50:
        print(f"\n… {len(lines) - 50} more candidate lines omitted")